# Z3 solver

## Z3 - osnove

In [1]:
from z3 import *

In [2]:
# Zapisati uslov da je 4-bitna reprezentacija broja palindrom, ali da nisu svi bitovi isti

A, B, C, D = Bools('A B C D')

s = Solver()
s.add(A == D, B == C, Not(And(A == B, B == C, C == D)))
if s.check() == sat:
    print(s.model())

[A = True, D = True, B = False, C = False]


In [3]:
# Za ispis svih resenja

A, B, C, D = Bools('A B C D')

s = Solver()
s.add(A == D, B == C, Not(And(A == B, B == C, C == D)))
while s.check() == sat:
    print(s.model())
    # Dodamo uslov da trenutno resenje nije jednako prethodnom, bez ovoga upada u beskonacnu petlju
    s.add(Not(And(A == s.model()[A], B == s.model()[B], C == s.model()[C], D == s.model()[D])))

[A = True, D = True, B = False, C = False]
[A = False, D = False, B = True, C = True]


In [4]:
x, y, z = Reals('x y z')

s = Solver()
s.add(
    x + 5*y - 3*z == 4,
    -x + y + z ==3 ,
    2*x + y - z == 1
)

if s.check() == sat:
    print(s.model())

[y = 7/4, x = 1/2, z = 7/4]


In [5]:
x, y, z = Reals('x y z')

s = Solver()
s.add(x > 1, y > 1, x + y >3, z - x < 10)
print(s.check())

m = s.model()
print(f'x = {m[x]}')

for d in m.decls():
    print(f'{d.name()} = {m[d]}')

sat
x = 3/2
y = 2
x = 3/2
z = 0


## Z3 - Uproscavanje izraza

In [6]:
x, y = Reals('x y')
solve(x + 10000000000000000000000 == y, y > 20000000000000000)

print(Sqrt(2) + Sqrt(3))
print(simplify(Sqrt(2) + Sqrt(3)).sexpr())
print((x + Sqrt(y) * 2).sexpr())

[y = 20000000000000001, x = -9999979999999999999999]
2**(1/2) + 3**(1/2)
(root-obj (+ (^ x 4) (* (- 10) (^ x 2)) 1) 4)
(+ x (* (^ y (/ 1.0 2.0)) 2.0))


In [7]:
x = Int('x')
y = Int('y')
print(simplify(x + y + 2*x +3))
print(simplify(x < y + x +2))
print(simplify(And(x + 1 >= 3, x**2 + x**2 + y**2 + 2 >= 5)))

3 + 3*x + y
Not(y <= -2)
And(x >= 2, 2*x**2 + y**2 >= 3)


## N - dama

In [10]:
# Bool p_i_j na polju (i, j) se nalazi dama
# Int Q_{i} broj kolone u kojoj se nalazi dama u i-tom redu
# Q_4 = 3 <=> U cetvrtom redu, dama se nalazi u trecoj koloni

n = 8
Q = [Int(f'Q_{i}') for i in range(n)]

val_c = [And(0 <= q, q<n) for q in Q]
col_c = [Distinct(Q)]
diag_c = [
    And(Q[i]-Q[j] != i - j, Q[i]-Q[j] != j-i)
    for i in range(n) for j in range(i) if i != j
]

n_queens = val_c + col_c + diag_c
solve(n_queens)

[Q_3 = 2,
 Q_1 = 5,
 Q_7 = 4,
 Q_5 = 3,
 Q_4 = 0,
 Q_0 = 1,
 Q_2 = 7,
 Q_6 = 6]


## Sudoku

In [11]:
# Promenljive za svako polje na tabli
X = [ [ Int(f'x_{i+1}_{j+1}') for j in range(9) ] for i in range(9) ]

# Svako polje moze imati vrednost izmedju 1 i 9
cell_values = [And(1 <= X[i][j], X[i][j] <=9) for i in range(9) for j in range(9)]

# Brojevi u redovima su jedinstveni
row_values = [ Distinct(X[i]) for i in range(9) ]

# Brojevi u kolonama su jedinstveni
col_values   = [ Distinct([ X[i][j] for i in range(9) ]) for j in range(9) ]

# Svaki podkvadrat je jedinstven
sq_c = []

for i0 in range(3):           # iterate over subgrid rows
    for j0 in range(3):       # iterate over subgrid columns
        subgrid = []
        for i in range(3):    # iterate over cells inside subgrid row
            for j in range(3):# iterate over cells inside subgrid column
                subgrid.append(X[3*i0 + i][3*j0 + j])
        sq_c.append(Distinct(subgrid))

sudoku_constraints = cell_values + row_values + col_values + sq_c

instance = ((0,0,0,0,9,4,0,3,0),
            (0,0,0,5,1,0,0,0,7),
            (0,8,9,0,0,0,0,4,0),
            (0,0,0,0,0,0,2,0,8),
            (0,6,0,2,0,1,0,5,0),
            (1,0,2,0,0,0,0,0,0),
            (0,7,0,0,0,0,5,2,0),
            (9,0,0,0,6,5,0,0,0),
            (0,4,0,9,7,0,0,0,0))

instance_c = [ If(instance[i][j] == 0, True, X[i][j] == instance[i][j]) 
               for i in range(9) for j in range(9) ]

s = Solver()
s.add(sudoku_constraints + instance_c)
if s.check() == sat:
    m = s.model()
    r = [ [ m.evaluate(X[i][j]) for j in range(9) ] 
          for i in range(9) ]
    print_matrix(r)
else:
    print("failed to solve")

[[7, 1, 5, 8, 9, 4, 6, 3, 2],
 [2, 3, 4, 5, 1, 6, 8, 9, 7],
 [6, 8, 9, 7, 2, 3, 1, 4, 5],
 [4, 9, 3, 6, 5, 7, 2, 1, 8],
 [8, 6, 7, 2, 3, 1, 9, 5, 4],
 [1, 5, 2, 4, 8, 9, 7, 6, 3],
 [3, 7, 6, 1, 4, 8, 5, 2, 9],
 [9, 2, 8, 3, 6, 5, 4, 7, 1],
 [5, 4, 1, 9, 7, 2, 3, 8, 6]]


## Z3 - Funkcije

In [12]:
x = Int('x')
y = Int('y')
f = Function('f', IntSort(), IntSort())
solve(f(f(x)) == x, f(x) == y, x != y)

[x = 0, y = 1, f = [1 -> 0, else -> 1]]


In [13]:
x = Int('x')
y = Int('y')
f = Function('f', IntSort(), IntSort())
s = Solver()
s.add(f(f(x)) == x, f(x) == y, x != y)
print(s.check())
m = s.model()
print("f(f(x)) =", m.evaluate(f(f(x))))
print("f(x)    =", m.evaluate(f(x)))

sat
f(f(x)) = 0
f(x)    = 1


## Predikatska logika

In [ ]:
# all humans are mortal
# Socrates is a human
# so Socrates mortal 
############################################

Object = DeclareSort('Object')

Human = Function('Human', Object, BoolSort())
Mortal = Function('Mortal', Object, BoolSort())

# a well known philosopher
socrates = Const('socrates', Object)

# free variables used in forall must be declared Const in python
x = Const('x', Object)

axioms = [ForAll([x], Implies(Human(x), Mortal(x))), 
          Human(socrates)]


solver = Solver()
solver.add(axioms)

if solver.check() == sat:
    conjecture = Mortal(socrates)
    solver.add(Not(conjecture))
    if solver.check() == unsat:
        print('Conjecture proved!')
else:
    print('contradicting axioms')
    
# classical refutation

In [15]:
# Dve nemimoilazne prave se seku ili su paralelne.
# Prave koje se seku leže u istoj ravni.
# Prave koje su paralelene leže u istoj ravni.
# Dve nemimoilazne prave leže u istoj ravni.

# m(X,Y) - X i Y su nemimoilazne: m: PxP -> B
# s(X,Y) - X i Y se seku: s: PxP -> B
# p(X,Y) - X i Y su paralelne: p: PxP -> B
# r(X,Y) - X i Y leze u istoj ravni: r: PxP -> B

B = BoolSort()
P = DeclareSort('Prave')

m = Function('m', P, P, B)
s = Function('s', P, P, B)
p = Function('p', P, P, B)
r = Function('r', P, P, B)

x, y = Consts('x y', P)

solver = Solver()
axioms = [
    ForAll([x,y], Implies(m(x,y), Or(s(x,y), p(x,y)))),
    ForAll([x,y], Implies(s(x,y), r(x,y))),
    ForAll([x,y], Implies(p(x,y), r(x,y))),
]
solver.add(axioms)

if solver.check() == sat:
    conjecture = ForAll([x,y], Implies(m(x,y), r(x,y)))
    solver.add(Not(conjecture))
    if solver.check() == unsat:
        print('Conjecture proved!')
    else:
        print('Contradicting axioms')

Conjecture proved!


In [20]:
# Svaka dva brata imaju zajedničkog roditelja.
# Roditelj je stariji od deteta.
# Postoje braća.
# Ni jedna osoba nije starija od druge.

B = BoolSort()
O = DeclareSort('Osoba')

b = Function('braca', O, O, B)
r = Function('roditelj', O, O, B)
s = Function('stariji', O, O, B)

x, y, z = Consts('x y z', O)

solver = Solver()
axioms = [
    ForAll([x, y], Exists([z], Implies(b(x,y), And(r(z,x), r(z,y))))),
    ForAll([x, y], Implies(r(x, y), s(x, y))),
    Exists([x,y], b(x,y))
]

if solver.check() == sat:
    conjecture = ForAll([x,y], Not(s(x,y)))
    solver.add(Not(conjecture))
    if solver.check() == unsat:
        print('Conjecture proved!')
else:
    print('Contradicting axioms')

## Sta Z3 ne moze da uradi?

U Z3 je moguce izraziti mnogo vise stvari nego sto on zapravo moze da uradi. Na primer, faktorisanje celih brojeva je osnova RSA kriptografije. Dok Z3 moze da faktorise cele brojeve, ne moze to da uradi za bilo koji iole veci broj.

In [22]:
x, y = Ints("x y")
pubkey = 3 * 7
solve(x*y == pubkey, x > 1, y > 1)

[x = 3, y = 7]


In [23]:
x, y = Ints("x y")
pubkey = 1000000993	* 1000001011
solve(x * y == pubkey, x > 1, y > 1)

failed to solve


## Dokazivanje u iskaznoj logici

In [25]:
p, q = Bools('p q')
print(And(p,q))
print(Or(p,q))
print(Xor(p,q))
print(Not(p,q))
print(Implies(p,q))
print(p == q)

my_true_thm = Implies(And(p, q), p)
prove(my_true_thm)

And(p, q)
Or(p, q)
Xor(p, q)
Not(p)
Implies(p, q)
p == q
proved


In [27]:
# De Morgan's Law p & q == ~ (~p | ~q)
# p -> q == ~ p | q
# Peirce's Law ((p -> q) -> p) -> p 
p,q = Bools('p q')

t1 = And(p, q) == Not(Or(Not(p), Not(q)))
prove(t1)

t2 = Implies(p, q) == Or(Not(p), q)
prove(t2)

t3 = Implies(Implies(Implies(p,q),p), p)
prove(t3)

proved
proved
proved


# Z3 - Vezbanje

In [1]:
from z3 import *

In [5]:
def check_for_solution(solver):
    if solver.check() == sat:
        print(solver.model())
    else:
        print('Nema resenja')

In [6]:
'''
U igri mines dimenzija 2x3 dobijena je sledeca konfiguracija
|1|A|C|
|1|B|2|
A,B,C su neotvorena polja, a brojevi oznacavaju broj mina u okolnim poljima.
Zapisati u iskaznoj logici uslove koji moraju da vaze.
'''

A, B, C = Bools('A B C')

conditions = [
    Xor(A, B),
    Or(And(And(A, B), Not(C)), And(And(A, C), Not(B)), And(And(B, C), Not(A)))
]

s = Solver()
s.add(conditions)
check_for_solution(s)

[A = True, B = False, C = True]


In [7]:
'''
Date su dve kutije A,B robot mora da stavi objekat u tacno jednu od njih.
'''

A, B = Bools('A B')
s = Solver()
s.add(Xor(A, B))
check_for_solution(s)

[A = True, B = False]


In [8]:
''' 
|A|B|
|C|D|
Zapisati uslov da se u tabeli 2x2 sa poljima A,B,C,D moze postaviti tacno jedan zeton u 
svakom redu
'''

A, B, C, D = Bools('A B C D')
s = Solver()
s.add([Xor(A, B), Xor(C, D)])
check_for_solution(s)

[A = True, D = False, B = False, C = True]


In [9]:
'''
Dva dvobitna broja se sabiraju i daju rezultat 3.
1+2
2+1
3+0
0+3
    A B
    C D
    ---
    1 1
'''

A, B, C, D = Bools('A B C D')
s = Solver()
s.add([Or(B, D), Or(A, C)])
check_for_solution(s)

[D = False, A = True, B = True, C = False]


In [10]:
'''
U iskoznoj logici zapisati da je 4 bitna reprezentacija broja palindrom ali da 
bitovi nisu jednaki
ABCD
'''

A, B, C, D = Bools('A B C D')
s = Solver()
conditions = [
    A == D,
    B == C,
    Not(And(C==D,And(A==B, B==C)))
]
s.add(conditions)
check_for_solution(s)

[A = False, D = False, B = True, C = True]


In [11]:
'''
Tri polja se boje crvenom ili plavom. 
Ukoliko je prvo crveno, druga dva moraju biti iste boje.
Ukoliko je drugo crveno, trece mora biti plavo.
'''

A, B, C = Bools('A B C')
s = Solver()
s.add(Implies(Not(A), B==C))
s.add(Implies(Not(B), C))
check_for_solution(s)

[A = True, B = True, C = False]


In [17]:
'''
    A
   / \\
  B - C
Temana trougla A,B,C se boje sa dve boje, pri tome ni jedan par temena ne moze imati istu boju.
'''

A, B, C = Bools('A B C')
s = Solver()
s.add(conditions)
conditions = [
    Xor(A, B),
    Xor(B, C),
    Xor(A, C)
]
check_for_solution(s)

Nema resenja


In [18]:
'''
|A|B|
|C|D|
Tabela 2x2 se boji crvenom ili plavom bojom.
Ako je polje A ofarbano crvenom onda barem jedno od ostalih polja mora biti plavo.
Ako je polje D ofarabno plavom onda barem dva ostala moraju biti crvena.
Ne smeju sva polja biti ofarabana istom bojom.
'''

A, B, C, D = Bools('A B C D')
s = Solver()
conditions = [
    Implies(Not(A), Or(B, C, D)),
    Implies(D, Or(Not(And(A, B)), Not(And(B, C)), Not(And(A,C)))),
    Not(And(A==B, B==C, C==D))
]
s.add(conditions)
check_for_solution(s)

[A = True, D = False, B = False, C = True]
